In [1]:
import math
from collections import Counter

In [2]:
documents = {
    "D1": "Business intelligence improves decision making",
    "D2": "Decision support systems assist managers",
    "D3": "Business analytics supports data driven decision making"
}


In [3]:
print("ORIGINAL DOCUMENTS:")
for k, v in documents.items():
    print(k, ":", v)
print("\n")

ORIGINAL DOCUMENTS:
D1 : Business intelligence improves decision making
D2 : Decision support systems assist managers
D3 : Business analytics supports data driven decision making




In [4]:
def preprocess(text):
    text = text.lower()

    # basic tokenization
    words = text.split()

    # simple stemming (manual mapping)
    stems = {
        "improves": "improve",
        "making": "make",
        "systems": "system",
        "supports": "support",
        "analytics": "analytic",
        "managers": "manager",
        "driven": "drive"
    }

    processed = []
    for w in words:
        processed.append(stems[w] if w in stems else w)

    return processed


processed_docs = {doc_id: preprocess(text) for doc_id, text in documents.items()}

print("PREPROCESSED DOCUMENTS:")
for k, v in processed_docs.items():
    print(k, ":", v)
print("\n")


PREPROCESSED DOCUMENTS:
D1 : ['business', 'intelligence', 'improve', 'decision', 'make']
D2 : ['decision', 'support', 'system', 'assist', 'manager']
D3 : ['business', 'analytic', 'support', 'data', 'drive', 'decision', 'make']




In [5]:
vocab = sorted(list(set([w for doc in processed_docs.values() for w in doc])))

print("VOCABULARY:")
print(vocab, "\n")

# Term frequencies (count)
tf_counts = {doc_id: Counter(words) for doc_id, words in processed_docs.items()}

print("TERM COUNTS (BOW):")
for k, v in tf_counts.items():
    print(k, ":", dict(v))
print("\n")

VOCABULARY:
['analytic', 'assist', 'business', 'data', 'decision', 'drive', 'improve', 'intelligence', 'make', 'manager', 'support', 'system'] 

TERM COUNTS (BOW):
D1 : {'business': 1, 'intelligence': 1, 'improve': 1, 'decision': 1, 'make': 1}
D2 : {'decision': 1, 'support': 1, 'system': 1, 'assist': 1, 'manager': 1}
D3 : {'business': 1, 'analytic': 1, 'support': 1, 'data': 1, 'drive': 1, 'decision': 1, 'make': 1}




In [6]:
def compute_tf(count, total_words):
    return count / total_words if total_words > 0 else 0

def compute_idf(term, docs):
    N = len(docs)
    df = sum(term in docs[d] for d in docs)
    return math.log(N / df)

# Terms required for full-step computation
target_terms = ["business", "make", "support", "data", "system"]

print("TARGET TERMS:", target_terms, "\n")

# Compute TF-IDF manually
tfidf = {term: {} for term in target_terms}

for term in target_terms:
    idf = compute_idf(term, processed_docs)
    for doc_id, words in processed_docs.items():
        count = tf_counts[doc_id][term]
        tf = compute_tf(count, len(words))
        tfidf[term][doc_id] = {
            "TF": tf,
            "IDF": idf,
            "TF-IDF": tf * idf
        }

TARGET TERMS: ['business', 'make', 'support', 'data', 'system'] 



In [7]:
print("===== STEP-BY-STEP TF, IDF, TF-IDF =====\n")

for term in target_terms:
    print(f"TERM: '{term}'")
    idf = tfidf[term]["D1"]["IDF"]
    print(f"  IDF = ln(3 / df) = {idf:.6f}")
    for doc_id in ["D1", "D2", "D3"]:
        tf = tfidf[term][doc_id]["TF"]
        tfidf_val = tfidf[term][doc_id]["TF-IDF"]
        print(f"  {doc_id}: TF={tf:.6f}  TF-IDF={tfidf_val:.6f}")
    print("\n")

===== STEP-BY-STEP TF, IDF, TF-IDF =====

TERM: 'business'
  IDF = ln(3 / df) = 0.405465
  D1: TF=0.200000  TF-IDF=0.081093
  D2: TF=0.000000  TF-IDF=0.000000
  D3: TF=0.142857  TF-IDF=0.057924


TERM: 'make'
  IDF = ln(3 / df) = 0.405465
  D1: TF=0.200000  TF-IDF=0.081093
  D2: TF=0.000000  TF-IDF=0.000000
  D3: TF=0.142857  TF-IDF=0.057924


TERM: 'support'
  IDF = ln(3 / df) = 0.405465
  D1: TF=0.000000  TF-IDF=0.000000
  D2: TF=0.200000  TF-IDF=0.081093
  D3: TF=0.142857  TF-IDF=0.057924


TERM: 'data'
  IDF = ln(3 / df) = 1.098612
  D1: TF=0.000000  TF-IDF=0.000000
  D2: TF=0.000000  TF-IDF=0.000000
  D3: TF=0.142857  TF-IDF=0.156945


TERM: 'system'
  IDF = ln(3 / df) = 1.098612
  D1: TF=0.000000  TF-IDF=0.000000
  D2: TF=0.200000  TF-IDF=0.219722
  D3: TF=0.000000  TF-IDF=0.000000




In [8]:
print("===== TF-IDF SUMMARY TABLE =====\n")
for term in target_terms:
    print(term.upper())
    for doc_id in ["D1", "D2", "D3"]:
        print(f"  {doc_id}: {tfidf[term][doc_id]['TF-IDF']:.6f}")
    print()

===== TF-IDF SUMMARY TABLE =====

BUSINESS
  D1: 0.081093
  D2: 0.000000
  D3: 0.057924

MAKE
  D1: 0.081093
  D2: 0.000000
  D3: 0.057924

SUPPORT
  D1: 0.000000
  D2: 0.081093
  D3: 0.057924

DATA
  D1: 0.000000
  D2: 0.000000
  D3: 0.156945

SYSTEM
  D1: 0.000000
  D2: 0.219722
  D3: 0.000000

